# 3 Paths after writing a prompt

1. Option 1: Test the prompt once and decide its good enough!
2. Option 2: Test the prompt a few times and tweak it to handle a corner case or two!
3. Option 3: Tun the prompt through an evaluation pipeline to score it then iterate on the prompt...

# Initial Prompt Draft
- initial prompt e.g. Please answer the users question : {question}

# Create an Eval Dataset
- possible questions
- sample input questions

# Feed Through Claude
- Responses

# Feed Through a Grader
- Average out all the numbers and take average score

# Change Prompt and Repeat
- may be add some extra details at the end of the prompt



In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    print(f">>>>{message}")
    return message.content[0].text

In [13]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```
    
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code
    
    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [16]:
dataset = generate_dataset()
print(f"\n dataset >>> {dataset}")

>>>>Message(id='msg_019FkguDR1FDAw3CpRj2vM2g', container=None, content=[TextBlock(citations=None, text='\n[\n    {\n        "task": "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., \'s3://my-bucket-us-west-2/path/to/file\'). The function should return the region code or None if not found."\n    },\n    {\n        "task": "Create a JSON object that represents an AWS IAM policy allowing a principal to read objects from a specific S3 bucket and list its contents."\n    },\n    {\n        "task": "Write a regular expression that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters, e.g., \'i-0abcd1234efgh5678\')."\n    }\n]\n', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='stop_sequence', stop_sequence='```', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_t

# Saving the dataset

In [17]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [19]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }
    

def run_eval(dataset):
    """Loads the json dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [20]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)


>>>>Message(id='msg_01FWWPjFUVkHZr7HvtNKbCYg', container=None, content=[TextBlock(citations=None, text='# AWS Region Extraction from S3 URI\n\nHere\'s a comprehensive solution with multiple approaches:\n\n## Solution 1: Simple Regex Pattern (Recommended)\n\n```python\nimport re\n\ndef extract_region_from_s3_uri(uri: str) -> str | None:\n    """\n    Extract AWS region from an S3 URI.\n    \n    Args:\n        uri: S3 URI in format \'s3://bucket-name/path\' or \'s3://bucket-name-region/path\'\n        \n    Returns:\n        Region code (e.g., \'us-west-2\') or None if not found\n    """\n    if not uri.startswith(\'s3://\'):\n        return None\n    \n    # Match AWS region pattern: us-east-1, eu-west-2, ap-southeast-1, etc.\n    region_pattern = r\'(us|eu|ap|ca|sa|me|af)-[a-z]+-\\d\'\n    match = re.search(region_pattern, uri)\n    \n    return match.group(0) if match else None\n```\n\n## Solution 2: Using boto3 (For Real S3 Buckets)\n\n```python\nimport boto3\nfrom botocore.exceptio

In [23]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extraction from S3 URI\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Simple Regex Pattern (Recommended)\n\n```python\nimport re\n\ndef extract_region_from_s3_uri(uri: str) -> str | None:\n    \"\"\"\n    Extract AWS region from an S3 URI.\n    \n    Args:\n        uri: S3 URI in format 's3://bucket-name/path' or 's3://bucket-name-region/path'\n        \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n    \"\"\"\n    if not uri.startswith('s3://'):\n        return None\n    \n    # Match AWS region pattern: us-east-1, eu-west-2, ap-southeast-1, etc.\n    region_pattern = r'(us|eu|ap|ca|sa|me|af)-[a-z]+-\\d'\n    match = re.search(region_pattern, uri)\n    \n    return match.group(0) if match else None\n```\n\n## Solution 2: Using boto3 (For Real S3 Buckets)\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef extract_region_from_s3_uri_boto3(uri: str) -> str | None:\n

# Graders

1. Code - Programmatically evaluate the result. Checking output length, verifying output does not have certain words, syntax validation, readability scores.
2. Model - Ask a model to assign a score to the output or compare 2 versions, response quality, completeness, safety, helpfulness
3. Human - Ask a human to assign a score to the output or compare 2 versions, usefull for depth, relevance, general response quality, comprehensiveness

In [25]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [26]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [27]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [28]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [29]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

>>>>Message(id='msg_013RFpK7wQiXywV8oahKyvY1', container=None, content=[TextBlock(citations=None, text='# AWS Region Extraction from S3 URI\n\nHere\'s a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(uri: str) -> Optional[str]:\n    """\n    Extract AWS region from an S3 bucket URI.\n    \n    Supports multiple S3 URI formats:\n    - s3://bucket-name-region/path\n    - s3://bucket-name/path (returns None)\n    - s3a://bucket-name-region/path\n    \n    Args:\n        uri: S3 URI string\n        \n    Returns:\n        Region code (e.g., \'us-west-2\') or None if not found\n        \n    Example:\n        >>> extract_region_from_s3_uri(\'s3://my-bucket-us-west-2/path/to/file\')\n        \'us-west-2\'\n    """\n    if not uri:\n        return None\n    \n    # Valid AWS region pattern\n    region_pattern = r\'(us|eu|ap|ca|sa|me)-(north|south|east|west|central)?-?\\d+\'\n    \n    # Extract bucket nam

In [30]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extraction from S3 URI\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(uri: str) -> Optional[str]:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI.\n    \n    Supports multiple S3 URI formats:\n    - s3://bucket-name-region/path\n    - s3://bucket-name/path (returns None)\n    - s3a://bucket-name-region/path\n    \n    Args:\n        uri: S3 URI string\n        \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n        \n    Example:\n        >>> extract_region_from_s3_uri('s3://my-bucket-us-west-2/path/to/file')\n        'us-west-2'\n    \"\"\"\n    if not uri:\n        return None\n    \n    # Valid AWS region pattern\n    region_pattern = r'(us|eu|ap|ca|sa|me)-(north|south|east|west|central)?-?\\d+'\n    \n    # Extract bucket name from S3 URI\n    # Handles s3://, s3a://, s3n://, https://, etc.\n    match = re.mat